# **Практика 6 (1). Визуализация**

В этой практике работаем с **почасовыми данными** Bike Sharing и тренируем графики на `matplotlib` и `seaborn`.


## Описание данных

Мы работаем с датасетом Bike Sharing - https://www.kaggle.com/datasets/lakshmi25npathi/bike-sharing-dataset

В этом датасете собраны данные о работе системы городского велопроката. Такие системы являются современной версией традиционного проката велосипедов - регистрация пользователя, аренда велосипеда и его возврат автоматизированы. Пользователь может взять велосипед в одной точке города и вернуть его в другой.

Системы велошеринга интересны не только как городской транспортный сервис, но и как источник данных для анализа городской мобильности. В отличие от автобусов или метро, где часто фиксируются только агрегированные пассажиропотоки, в велопрокате можно явно наблюдать время поездки, момент аренды и возврата велосипеда, а также связанные с этим погодные и календарные условия.

Поэтому данные велошеринга можно рассматривать как своеобразную «сенсорную сеть» города. По ним можно анализировать поведение жителей, сезонность спроса, влияние погоды, различия между буднями и выходными, а также потенциально замечать важные городские события.

В датасете есть два файла:

- `hour.csv` — данные по часам;
- `day.csv` — данные по дням.

Оба файла содержат почти одинаковые признаки. Отличие в том, что в `day.csv` нет признака `hr`, потому что данные уже агрегированы по дням.

| Признак | Описание |
|---|---|
| `instant` | порядковый номер записи |
| `dteday` | дата наблюдения |
| `season` | сезон: `1` — весна, `2` — лето, `3` — осень, `4` — зима |
| `yr` | год: `0` — 2011, `1` — 2012 |
| `mnth` | месяц от `1` до `12` |
| `hr` | час от `0` до `23`; есть только в `hour.csv` |
| `holiday` | является ли день праздничным: `1` — да, `0` — нет |
| `weekday` | день недели |
| `workingday` | рабочий день: `1`, если день не является выходным или праздником; иначе `0` |
| `weathersit` | погодная ситуация |
| `temp` | нормализованная температура в градусах Цельсия |
| `atemp` | нормализованная ощущаемая температура в градусах Цельсия |
| `hum` | нормализованная влажность |
| `windspeed` | нормализованная скорость ветра |
| `casual` | количество незарегистрированных пользователей |
| `registered` | количество зарегистрированных пользователей |
| `cnt` | общее количество арендованных велосипедов: `casual + registered` |


**Расшифровка признака `weathersit`**

| Значение | Погода |
|---|---|
| `1` | ясно, небольшая облачность, переменная облачность |
| `2` | туман, облачно, небольшая облачность |
| `3` | небольшой снег, небольшой дождь, гроза, рассеянная облачность |
| `4` | сильный дождь, град, гроза, туман, снег |

**Нормализация погодных признаков**

Некоторые погодные признаки в датасете представлены в нормализованном виде.

- `temp` — температура нормализована по шкале от `-8` до `+39` градусов Цельсия.
- `atemp` — ощущаемая температура нормализована по шкале от `-16` до `+50` градусов Цельсия.
- `hum` — влажность разделена на `100`.
- `windspeed` — скорость ветра разделена на `67`.

Чтобы вернуть значения к более привычному виду, можно использовать следующие преобразования:

```python
    df["temp_c"] = df["temp"] * 47 - 8
    df["atemp_c"] = df["atemp"] * 66 - 16
    df["humidity_pct"] = df["hum"] * 100
    df["windspeed"] = df["windspeed"] * 67
```

In [1]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# будем работать с почасовыми данными
bike_hour = pd.read_csv("../data/hour.csv")

bike_hour.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


## **1. Предобработка**


>*Посмотрите основную информацию о датасете и ответьте на вопросы:*
>- сколько наблюдений и признаков?
>- сколько пропущенных значений по каждому признаку?
>- какой тип у каждого признака?
>- какие признаки числовые, а какие категориальные?


>*Приведите признак `season` в естественно-язычный вид. Соответствие признаков и значений можно посмотреть в описании датасета. Новый признак назовите `season_name`.*


>*Приведите признак `weathersit` в естественно-язычный вид. Новый признак назовите `weather_name`.*


>*Приведите признак `weekday` в естественно-язычный вид. Новый признак назовите `weekday_name`.*


>*Приведите признак `workingday` в естественно-язычный вид. Новый признак назовите `workingday_name`.*


Важные целевые признаки:

- `casual` — количество незарегистрированных пользователей;
- `registered` — количество зарегистрированных пользователей;
- `cnt` — общее количество арендованных велосипедов.

Проверьте, что:

```python
cnt = casual + registered
```


Погодные признаки в датасете нормализованы.

Верните их к более понятному виду:

- `temp_c` — температура в градусах Цельсия;
- `atemp_c` — ощущаемая температура;
- `humidity_pct` — влажность в процентах;
- `windspeed_real` — скорость ветра.

В описании датасета температура нормализована по формуле min-max. Поэтому используйте обратное преобразование.


Так как мы работаем с `hour.csv`, одна строка — это один час, а не один день.

Для некоторых графиков нам понадобится дневная агрегация. Создадим таблицу `bike_day_from_hour`, где для каждого дня посчитаны аггрегированные признаки.
- для `cnt`, `casual`, `registered` - суммируем
- для `temp_c`, `atemp_c`, `humidity_pct`, `windspeed_real` - усредняем
- для `season_name`, `weather_name`, `weekday_name`, `workingday_name` - берем самое популярное значение
- `mnth` и `yr` - можно взять любое 

## **2. Визуализация**

Дальше будем тренировать разные типы графиков.

Важно: перед графиком всегда задаем себе вопрос — **что именно мы хотим увидеть?**

- динамику во времени → какой график?
- распределение → какой график?
- сравнение категорий → какой график?
- связь двух числовых признаков → какой график?

### **2.1 Динамика спроса**

Посмотрим на **динамику спроса по месяцам**: как менялось общее число аренд велосипедов.

1. Сгруппируйте данные по месяцам:
    - создайте новый признак `dtemonth`;
    - для каждого месяца посчитайте суммарное количество аренд `cnt`.


2. Нарисуйте график с помощью `matplotlib`: подпишите график и оси.


### **2.2 Распределение спроса**

В разные дни может быть разное количество аренд. Мы можем собрать дневные суммы и посмотреть распределение дневного спроса.

Гистограмма отвечает на вопрос: **какие значения спроса встречаются часто, а какие редко?**


1. Нарисуйте распределение дневного количества аренд с помощью `matplotlib`.


2. Теперь сравним распределение спроса при разной погоде. Используем `seaborn` и `boxplot`.

`boxplot` удобен, когда нужно сравнить распределение числового признака между категориями.


### **2.3 Сравнение спроса между сезонами**

1. Посчитайте средний дневной спрос по сезонам и нарисуйте `barplot` с помощью `matplotlib`.


2. Теперь сделайте похожий график через `seaborn`: сравните средний спрос по дням недели.


3. Добавьте разбиение по типу дня: рабочий день или выходной/праздник.


### **2.4 Сравнение распределений дневного спроса

`barplot` показывает среднее, но может скрывать разброс.

Если нам важно увидеть не только среднее, но и форму распределения, лучше использовать:

- `boxplot` — медиана, квартили, выбросы;
- `violinplot` — форма распределения и плотность значений.


1. Сравните распределение дневного спроса по сезонам с помощью `boxplot`.


2. Постройте `violinplot` для спроса в рабочие и нерабочие дни.


### **2.5 Анализ почасового спроса**

Так как у нас есть почасовые данные, можно посмотреть, в какие часы люди чаще всего берут велосипеды.

Для этого сгруппируем данные по `hr` и посчитаем среднее количество аренд.


Теперь сравним почасовой профиль спроса в рабочие и нерабочие дни.

Если в рабочие дни есть пики утром и вечером, это может быть похоже на поездки на работу или учебу.


Сравним почасовой спрос для `casual` и `registered` пользователей.


### **2.6 От чего может зависеть числовой спрос?**

1. Постройте связь температуры и дневного спроса.


2. Добавьте больше информации на график:

- `hue` — сезон;
- `size` — влажность;
- `style` — тип дня.


### **2.7 Спрос на шеринг по дням недели и часам**

Сделайте группирову
- строки — дни недели;
- столбцы — часы;
- значения - среднее количество аренд

Визуализируйте эти данные

Сделайте другую группировку: средний спрос по месяцам и часам.

Так можно увидеть не только сезонность, но и то, меняется ли почасовой паттерн в разные месяцы.

Визуализируйте эти данные

### **2.8 Корреляции**

Корреляция показывает, насколько два числовых признака линейно связаны между собой.

Значения корреляции:

- ближе к `1` — сильная положительная связь;
- ближе к `-1` — сильная отрицательная связь;
- около `0` — линейной связи почти нет.


Постройте корреляционную матрицу

Визуализируйте ее

### **2.9 Тренируем subplots**

`subplots` помогает собрать несколько связанных графиков в одну фигуру.

Например, сравним распределения нескольких числовых признаков.


## **3. Мини-исследование**

Теперь нужно самостоятельно ответить на несколько аналитических вопросов.

Для каждого вопроса:

1. выберите подходящий график;
2. постройте его;
3. подпишите заголовок и оси;
4. после графика напишите короткий вывод.


### Вопрос 1. Когда спрос максимальный?

Постройте график среднего количества аренд по часам. Дополнительно разделите график на рабочие и нерабочие дни.


In [ ]:
# ваш код здесь


**Вывод:**

Напишите здесь 2-3 предложения по графику.


### Вопрос 2. Чем отличаются casual и registered users?

Сравните почасовой профиль `casual` и `registered` пользователей. Подумайте, какая группа больше похожа на регулярные поездки, а какая — на досуг.


In [ ]:
# ваш код здесь


**Вывод:**

Напишите здесь 2-3 предложения по графику.


### Вопрос 3. Как влияет погода?

Постройте график, который показывает распределение дневного спроса при разной погоде.


In [ ]:
# ваш код здесь


**Вывод:**

Напишите здесь 2-3 предложения по графику.


### Вопрос 4. Есть ли сезонность?

Сравните спрос по месяцам и сезонам. Можно использовать lineplot, barplot или boxplot.


In [ ]:
# ваш код здесь


**Вывод:**

Напишите здесь 2-3 предложения по графику.


### Вопрос 5. Отличается ли поведение в рабочие и выходные дни?

Постройте почасовой профиль спроса отдельно для рабочих и нерабочих дней.


In [ ]:
# ваш код здесь


**Вывод:**

Напишите здесь 2-3 предложения по графику.
